# drive2sim ingest từ $source

Notebook do `d2s kaggle ingest` sinh ra (CONTRACTS mục 9.4), chỉ dùng CPU. Nó chạy `d2s scene ingest` trên dữ liệu gốc gắn từ `$source` và giữ cảnh ở `/kaggle/working/data/<scene_id>/`; notebook train gắn output này thay vì upload cảnh.

In [ ]:
%%bash
# Lấy đúng commit của code trên máy. Ingest không cần submodule (mã tiền xử lý nằm trong package).
set -euo pipefail
git clone --quiet --branch $branch $repo_url /kaggle/tmp/repo
cd /kaggle/tmp/repo
git checkout --quiet $commit


In [ ]:
%%bash
# Môi trường nhẹ cho ingest, cùng phiên bản với môi trường chính ở những gói ingest dùng: Python 3.10,
# numpy 1.23.5 (requirements.txt), PyTorch 2.4.1 bản CPU (mã tiền xử lý import torch), devkit của
# nuScenes. Converter KITTI-360 viết bằng Python thuần.
set -euo pipefail
command -v uv >/dev/null || python -m pip install --quiet uv
uv venv --quiet --seed --python 3.10 /kaggle/tmp/venv
uv pip install --quiet --python /kaggle/tmp/venv/bin/python \
  --index-url https://pypi.org/simple --extra-index-url https://download.pytorch.org/whl/cpu \
  -e /kaggle/tmp/repo "numpy==1.23.5" "torch==2.4.1" "nuscenes-devkit==1.2.0" \
  "opencv-python-headless==4.11.0.86" "pyquaternion==0.9.9" "matplotlib<3.9" tqdm pillow
/kaggle/tmp/venv/bin/python -c "import numpy, torch; print('numpy', numpy.__version__, 'torch', torch.__version__)"


In [ ]:
# Tìm nguồn dưới /kaggle/input theo tên (không giả định đường gắn của Kaggle), chạy ingest, rồi chỉ
# giữ data/<scene_id>/ và ingest_result.json trong output.
import json
import pathlib
import shutil
import subprocess

input_root = pathlib.Path("/kaggle/input")
name, sub = "$source_name", "$source_path"
found = [p for depth in ("", "*/", "*/*/") for p in input_root.glob(depth + name) if p.is_dir()]
assert len(found) == 1, f"tìm thấy {len(found)} thư mục {name} dưới {input_root}: {found}"
raw = found[0] / sub if sub else found[0]
print("nguồn", raw)
working = pathlib.Path("/kaggle/working")
command = ["/kaggle/tmp/venv/bin/d2s", "scene", "ingest", str(raw), "--workdir", str(working)]
command += json.loads(r"""$ingest_args""")
print(" ".join(command), flush=True)
code = subprocess.run(command, cwd="/kaggle/tmp/repo").returncode
scenes = sorted(p.parent for p in (working / "data").glob("*/manifest.yaml"))
for item in working.iterdir():
    if item.name != "data":
        shutil.rmtree(item) if item.is_dir() else item.unlink()
# Mã 1 là cảnh đã ghi nhưng báo cáo kiểm tra có lỗi: vẫn giữ cảnh để xem báo cáo trên máy.
assert code in (0, 1) and len(scenes) == 1, f"ingest lỗi (mã {code}), cảnh: {scenes}"
result = {"scene_id": scenes[0].name, "commit": "$commit", "exit_code": code}
result.update(json.loads(r"""$signature"""))  # nguồn, --path, tham số: để lần sau so
(working / "ingest_result.json").write_text(json.dumps(result, indent=2))
print(result)
# In file nhỏ của cảnh vào log (stderr của tiến trình, có trong `kaggle kernels logs`), để máy
# đọc bằng một lệnh API thay vì liệt kê hàng chục nghìn file của output (CONTRACTS mục 9.4).
import base64
import os

scene = scenes[0]
# Tập file nhỏ (manifest, reports, pose, calib) lấy từ drive2sim/kaggle/scene_files.py của repo
# vừa clone, để máy xem được model mà không cần ảnh, LiDAR.
import sys

sys.path.insert(0, "/kaggle/tmp/repo")
from drive2sim.kaggle.scene_files import light_scene_files, write_log_lines

light = {"result": result, "files": light_scene_files(scene)}
write_log_lines("D2S_INGEST_LIGHT", light)  # dòng ngắn đánh số, stdout không chen vào được